# Cross-Dataset Evaluation

This notebook evaluates the fake-news DistilBERT model trained on the Kaggle fake-news dataset against an external dataset. The goal is to test generalization, not to retrain the model.

Primary research question: How well do transformer-based models generalize across fake-news datasets?

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.evaluation import build_predictions_frame, compute_metrics, create_run_directory, save_evaluation_outputs
from utils.model_loader import load_fake_news_model
from utils.prediction import predict_text
from utils.project_config import ARTIFACTS_DIR, DATA_DIR

pd.set_option("display.max_colwidth", 180)

## Load External FakeNewsNet Data

FakeNewsNet files are split into fake and real source files. This notebook uses titles because the local files do not include full article bodies.

In [ ]:
def load_fakenewsnet_titles(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    base_dir = data_dir / "FakeNewsNet"
    frames = []
    for path, label in [
        (base_dir / "gossipcop_fake.csv", "FAKE"),
        (base_dir / "politifact_fake.csv", "FAKE"),
        (base_dir / "gossipcop_real.csv", "REAL"),
        (base_dir / "politifact_real.csv", "REAL"),
    ]:
        frame = pd.read_csv(path)
        frame = frame.assign(label=label, source_file=path.name)
        frames.append(frame)

    full_frame = pd.concat(frames, ignore_index=True)
    full_frame["input_text"] = full_frame["title"].fillna("").astype(str).str.strip()
    full_frame = full_frame.loc[full_frame["input_text"].str.len() > 0].copy()
    return full_frame.reset_index(drop=True)


external_frame = load_fakenewsnet_titles()
external_frame["label"].value_counts()

## Evaluate Without Retraining

Set `LIMIT = None` for the full run. Start with a smaller limit when checking that model artifacts load correctly.

In [ ]:
LIMIT = None
labels = ["FAKE", "REAL"]
evaluation_frame = external_frame.copy()
if LIMIT is not None:
    evaluation_frame = evaluation_frame.sample(n=min(LIMIT, len(evaluation_frame)), random_state=42).reset_index(drop=True)

model, tokenizer = load_fake_news_model()

predicted_indices = []
predicted_labels = []
probabilities = []

for text in evaluation_frame["input_text"].tolist():
    predicted_index, probability_vector, _ = predict_text(model, tokenizer, text)
    predicted_indices.append(int(predicted_index))
    probabilities.append([float(value) for value in probability_vector])
    predicted_labels.append(labels[int(predicted_index)])

predictions = build_predictions_frame(
    examples=evaluation_frame,
    true_labels=evaluation_frame["label"].tolist(),
    predicted_labels=predicted_labels,
    predicted_indices=predicted_indices,
    probabilities=probabilities,
    labels=labels,
)

metrics = compute_metrics(
    true_labels=predictions["true_label"].tolist(),
    predicted_labels=predictions["predicted_label"].tolist(),
    probabilities=probabilities,
    labels=labels,
)

metrics

## Save Cross-Dataset Results

In [ ]:
output_dir = create_run_directory(ARTIFACTS_DIR, "cross_dataset", "fakenewsnet_titles")
save_evaluation_outputs(
    output_dir=output_dir,
    predictions=predictions,
    metrics=metrics,
    true_labels=predictions["true_label"].tolist(),
    predicted_labels=predictions["predicted_label"].tolist(),
    probabilities=probabilities,
    labels=labels,
)

output_dir

## Interpretation

Interpret the result as a domain-shift check. FakeNewsNet titles are shorter than the Kaggle title-plus-body training inputs, so weaker performance may reflect input-field mismatch, source/domain shift, or both. Report accuracy and macro F1 together, then inspect false positives and false negatives in the error-analysis notebook.